# NFCorpus Retrieval Benchmark
Real BM25, Dense and Hybrid results evaluated against official NFCorpus test qrels.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'artifacts/external-benchmarks/nfcorpus/benchmark_results.json').exists())
data = json.loads((ROOT / 'artifacts/external-benchmarks/nfcorpus/benchmark_results.json').read_text())
by_query = {row['query_id']: row for row in data['queries']}
print(data['dataset'], data['question_count'], 'questions,', data['document_count'], 'documents,', data['chunk_count'], 'chunks')

NFCorpus test 323 questions, 3633 documents, 4229 chunks


In [2]:
display(pd.DataFrame(data['summary']).T.style.format('{:.4f}'))

,ndcg_at_10,mrr_at_10,recall_at_5,recall_at_10,recall_at_20,recall_at_50,recall_at_100,hit_at_5,hit_at_10,hit_at_20
bm25,0.3088,0.5199,0.1205,0.1511,0.1792,0.2122,0.2439,0.6285,0.6873,0.7183
dense,0.3355,0.5338,0.1232,0.1604,0.1934,0.2457,0.2969,0.6285,0.7059,0.7616
rrf,0.3461,0.5535,0.1346,0.1685,0.1999,0.2486,0.2935,0.6811,0.7307,0.7709
reranked,0.2507,0.4187,0.0809,0.1338,0.1999,0.2486,0.2935,0.5759,0.6780,0.7709


In [3]:
def show_question(query_id):
    row = by_query[query_id]
    display(Markdown(f"## {query_id}\n{row['question']}"))
    display(pd.DataFrame([{'document_id': k, 'relevance': v} for k, v in row['gold_documents'].items()]).sort_values('relevance', ascending=False))
    for method, result in row['methods'].items():
        display(Markdown(f"### {method.upper()} — {result['latency_ms']:.2f} ms"))
        display(pd.DataFrame({'rank': range(1, len(result['ranking']) + 1), 'document_id': result['ranking'], 'gold_relevance': [row['gold_documents'].get(doc, 0) for doc in result['ranking']]}))
        display(pd.DataFrame([result['metrics']]))

query_ids = list(by_query)
show_question(query_ids[0])

## PLAIN-2
Do Cholesterol Statin Drugs Cause Breast Cancer?

,document_id,relevance
0,MED-2427,2
2,MED-2429,2
3,MED-2430,2
4,MED-2431,2
5,MED-14,2
6,MED-2432,2
1,MED-10,2
15,MED-3597,1
22,MED-4829,1
21,MED-4828,1


### BM25 — 8.51 ms

,rank,document_id,gold_relevance
0,1,MED-14,2
1,2,MED-2429,2
2,3,MED-10,2
3,4,MED-2428,1
4,5,MED-1193,0
...,...,...,...
95,96,MED-4787,0
96,97,MED-3860,0
97,98,MED-4714,0
98,99,MED-2774,0


,ndcg_at_10,mrr_at_10,recall_at_5,recall_at_10,recall_at_20,recall_at_50,recall_at_100,hit_at_5,hit_at_10,hit_at_20
0,0.79771,1.0,0.166667,0.333333,0.416667,0.625,0.625,1.0,1.0,1.0


...

,rank,document_id,gold_relevance
0,1,MED-14,2
1,2,MED-2429,2
2,3,MED-10,2
3,4,MED-2431,2
4,5,MED-2428,1
...,...,...,...
95,96,MED-4532,0
96,97,MED-2885,0
97,98,MED-1825,0
98,99,MED-4899,0


,ndcg_at_10,mrr_at_10,recall_at_5,recall_at_10,recall_at_20,recall_at_50,recall_at_100,hit_at_5,hit_at_10,hit_at_20
0,0.849935,1.0,0.208333,0.375,0.458333,0.583333,0.75,1.0,1.0,1.0


### RERANKED — 32.81 ms

,rank,document_id,gold_relevance
0,1,MED-1193,0
1,2,MED-2439,1
2,3,MED-2440,1
3,4,MED-2103,0
4,5,MED-2428,1
...,...,...,...
95,96,MED-4532,0
96,97,MED-2885,0
97,98,MED-1825,0
98,99,MED-4899,0


,ndcg_at_10,mrr_at_10,recall_at_5,recall_at_10,recall_at_20,recall_at_50,recall_at_100,hit_at_5,hit_at_10,hit_at_20
0,0.425934,0.5,0.125,0.333333,0.458333,0.583333,0.75,1.0,1.0,1.0


In [4]:
try:
    import ipywidgets as widgets
    widgets.interact(show_question, query_id=widgets.Dropdown(options=query_ids, description='Question'))
except ImportError:
    print('Call show_question(query_ids[0]) manually.')

interactive(children=(Dropdown(description='Question', options=('PLAIN-2', 'PLAIN-12', 'PLAIN-23', 'PLAIN-33',…

## Interpretation
Higher is better for every metric. Pure RRF (`k=60`) is strongest at top-10 and is the provisional default. Dense E5 remains strongest at Recall@100. The separate bi-encoder reranking experiment harms top-10 quality and must not be promoted. This external benchmark does not replace topic-relevance judgments for MedGuide's own supplement questions.